# Context Management and reliability

A customer says: "This is absolutely unacceptable! I've been waiting three weeks for a refund and nobody has helped me!" They have not asked to speak to a human. The agent can verify the refund status and initiate a resolution. What should the agent do?
A.Acknowledge frustration, investigate the refund status, and offer a resolution
B.Immediately escalate to a human agent because the customer sounds frustrated
C.Apologize for the delay and close the ticket, asking them to call support instead
D.Ask the customer to rate their frustration level on a scale of 1-10 first
✓ Correct — Explanation
Frustration alone is not an escalation trigger. The escalation rule is: explicit request for a human = immediate escalation; frustration without an explicit request = acknowledge and attempt resolution. In this case, the agent should validate the customer's experience ("I understand how frustrating a three-week wait is") and then address the underlying issue — verifying the refund status and initiating resolution if possible. Immediate escalation on frustration alone (A) ignores cases the agent can and should handle. Deflecting to a phone line (C) abandons the customer. Asking them to rate frustration (D) adds friction without value.



Production metrics show that when your agent resolves complex cases involving billing disputes or multi-order returns, customer satisfaction scores are 15% lower than for simple cases—even when the resolution is technically correct. Root cause analysis reveals the agent provides accurate resolutions but inconsistently explains the reasoning: sometimes omitting relevant policy details, other times missing timeline information or next steps. The specific context gaps vary by case. You want to improve resolution quality without adding human review overhead. Which approach is most effective?
A.Add a self-critique step where the agent evaluates its draft response for completeness—ensuring it addresses the customer's concern, includes relevant context, and anticipates follow-up questions.
B.Add a confirmation step where the agent asks "Does this fully address your concern?" before closing, letting customers request additional information if needed.
C.Implement few-shot examples in the system prompt showing complete resolution explanations for five common complex case types, demonstrating how to include policy context, timelines, and next steps.
D.Increase the model tier from Haiku to Sonnet for complex cases, routing based on detected case complexity.
✗ Incorrect — Explanation
The self-critique step (evaluator-optimizer pattern) directly addresses the root cause: gaps in completeness that vary by case. Before presenting the response, the agent evaluates its own draft against explicit criteria — does it explain the policy, include relevant timelines, address all concerns, and anticipate follow-up questions? This catch-mechanism is case-aware and handles the variability that makes few-shot examples insufficient. The confirmation question (B) shifts the burden to customers to identify their own information gaps. Few-shot examples (C) cover five specific templates but miss novel gap patterns in different case types. A model tier upgrade (D) may improve general quality but does not add a targeted completeness check.



Why is self-reported LLM confidence a poor signal for escalation routing?
A.The API does not expose a confidence score parameter for routing decisions
B.Confidence scores are only available when using chain-of-thought prompting
C.Self-reported confidence is reliable but too slow to compute for real-time use
D.LLMs are poorly calibrated — often highly confident on exactly wrong answers
✓ Correct — Explanation
LLM confidence calibration is poor, particularly for hard cases. Models often express high confidence on questions they answer incorrectly — the specific situations where escalation matters most. Escalation routing must use programmatic signals: missing required data fields, explicit error returns, policy threshold violations, or structured uncertainty flags — not Claude's self-assessment.



Your agent processes a 200-page research report, extracting findings from each section. After processing, the agent's synthesis consistently omits findings from sections 60–140 but reliably includes content from the first 30 pages and the final 20 pages. What is the root cause and correct fix?
A.Context window is too small; switch to a model with a larger context window to fit the full document
B.Extraction prompt is too general; add instructions targeting sections 60-140
C.Lost-in-the-middle effect — use per-section passes with a separate integration pass
D.Token limits are hit mid-document; implement chunking to process in segments
✓ Correct — Explanation
The pattern (reliable at beginning and end, gaps in the middle) is the hallmark of the "lost in the middle" effect — a well-documented phenomenon where models attend reliably to the start and end of long contexts but give less attention to middle content. The fix has two components: (1) place key findings summaries at the beginning of aggregated inputs, and (2) use per-section passes (each section gets its own focused context window) with a separate integration pass over all summaries. A larger context window (A) does not fix attention distribution — it just moves the diluted zone.


The web search subagent times out while researching a complex topic. You need to design how this failure information flows back to the coordinator agent. Which error propagation approach best enables intelligent recovery?
A.Return structured error context to the coordinator including the failure type, the attempted query, any partial results, and potential alternative approaches
B.Implement automatic retry logic with exponential backoff within the subagent, returning a generic "search unavailable" status only after all retries are exhausted
C.Catch the timeout within the subagent and return an empty result set marked as successful
D.Propagate the timeout exception directly to a top-level handler that terminates the entire research workflow
✓ Correct — Explanation
Structured error context gives the coordinator the information it needs to make intelligent recovery decisions — whether to retry with a modified query, try an alternative approach, or proceed with partial results. Option B's generic status hides valuable context from the coordinator, preventing informed decisions. Option C suppresses the error by marking failure as success, which prevents any recovery and risks incomplete research outputs. Option D terminates the entire workflow unnecessarily when recovery strategies could succeed.


A multi-turn conversation agent is losing context from early in long conversations — users have to repeat context they provided 20+ turns ago. What strategy best addresses this?
A.Increase the context window to fit more conversation turns in total
B.Delete old turns automatically when the context is nearly full
C.Ask users to restate their full context every 10 turns as a manual conversation refresh mechanism
D.Implement rolling summaries that compress earlier turns into structured context
✗ Incorrect — Explanation
Rolling summaries (also called context compression or memory summarization) maintain the most important information from earlier conversation turns in a compact form. Instead of dropping or truncating old turns, the agent periodically summarizes them ("User is researching competitor pricing for their enterprise software pitch to FinanceCo") and injects that summary as persistent context. This preserves semantic continuity without consuming the full context window.


A medical triage agent routes patient cases to either a nurse or a specialist based on its confidence assessment. Engineering review finds that 8% of high-confidence specialist referrals were cases the nurse could have handled, and 3% of high-confidence nurse referrals needed a specialist. How should the routing architecture be redesigned?
A.Lower the confidence threshold for specialist referral to be more conservative
B.Replace self-reported confidence with rule-based routing: symptom categories, vital sign thresholds, medication interaction flags from structured extraction — with a `requires_specialist` field derived from the data, not Claude's assessment
C.Add a second Claude model to validate routing decisions
D.Increase temperature to generate more routing decision diversity
✓ Correct — Explanation
Medical routing based on LLM confidence is a safety risk. The correct architecture uses objective, clinically-defined routing rules derived from structured data extraction: "fever > 39.5°C → specialist flag," "chest pain with shortness of breath → specialist flag," "medication interaction detected → specialist flag." These rules are encoded in the extraction schema and routing logic — not in Claude's self-assessment. The extracted structured data drives the decision.



Your agent achieves 55% first-contact resolution, well below the 80% target. Logs show it escalates straightforward cases (standard damage replacements with photo evidence) while attempting to autonomously handle complex situations requiring policy exceptions. What's the most effective way to improve escalation calibration?
A.Add explicit escalation criteria to your system prompt with few-shot examples demonstrating when to escalate versus resolve autonomously
B.Have the agent self-report a confidence score (1–10) before each response and automatically route requests to humans when confidence falls below a threshold
C.Deploy a separate classifier model trained on historical tickets to predict which requests need escalation before the main agent begins processing
D.Implement sentiment analysis to detect customer frustration levels and automatically escalate when negative sentiment exceeds a threshold
✓ Correct — Explanation
Adding explicit escalation criteria with few-shot examples directly addresses the root cause: unclear decision boundaries. This is the proportionate first response before adding infrastructure. Option B fails because LLM self-reported confidence is poorly calibrated — the agent is already incorrectly confident on hard cases. Option C is over-engineered, requiring labeled data and ML infrastructure when prompt optimization hasn't been tried. Option D solves a different problem entirely; sentiment doesn't correlate with case complexity, which is the actual issue.



Your research coordinator is producing synthesis reports, but the sources in the final report cannot be traced back to specific documents — the synthesis subagent merged everything into unsourced paragraphs. You are now asked to add citation tracking. What change makes this possible?
A.Ask the synthesis subagent to "add citations where possible"
B.Redesign the coordinator to pass structured context: each finding tagged with source ID, document title, URL, and page number; require the synthesis subagent to output a `citations` array alongside each claim
C.Add a post-synthesis citation extraction step that parses the report for URLs
D.Store all source documents in a shared database that the synthesis subagent queries
✓ Correct — Explanation
Citation tracking must be architected upstream, not retrofitted downstream. The coordinator must pass structured context (source ID → document title + URL + page) to the synthesis subagent, and the synthesis schema must include a `citations` field (array of source IDs) alongside each claim. The synthesis subagent can only attribute claims to sources if it received source metadata. Post-hoc citation extraction from unstructured prose is fragile and will miss many citations.


A customer support agent handles multi-turn conversations about billing disputes. After 15+ turns, the agent starts misremembering the exact disputed amount — stating "$43" instead of the customer's stated "$43.27" — and loses track of which charges the customer already confirmed. What is the most robust architectural fix?
A.Use /compact to summarize the conversation before it gets long
B.Increase max_tokens so the full conversation fits in a single prompt
C.Extract transactional facts (disputed amount, confirmed charges, account ID, timestamps) into a persistent "case facts" block that is included verbatim in every subsequent prompt, outside the summarized conversation history
D.Ask the customer to restate the disputed amount at the start of each turn
✓ Correct — Explanation
Progressive summarization loses precision on numerical values — "$43.27" becomes "$43" or "approximately $40." The fix is to extract transactional facts into a dedicated persistent block that bypasses summarization: amounts, dates, order numbers, statuses are stored structurally and injected verbatim into every prompt. The conversation history can be compressed but the fact block remains accurate. /compact (A) worsens the problem by further compressing numerical details. More tokens (B) delays but doesn't solve the problem. Re-asking customers (D) is a bad user experience and doesn't scale.


Your support agent handles 10,000 tickets per day. Analysis shows 12% are escalated to human agents. Post-escalation review finds: 60% of escalations were based on Claude's "I'm not fully confident" output, and 40% of those did not actually need human handling. Engineering is asked to reduce unnecessary escalations without increasing missed escalations. What is the correct architectural change?
A.Fine-tune the model with examples of correctly handled complex cases
B.Remove escalation entirely and let the model always attempt resolution
C.Replace confidence-based escalation with rule-based escalation: define specific triggers (refund > $500, unverifiable account, 3+ tool failures, specific issue categories) in the routing logic derived from structured extraction
D.Add a second call to Claude asking it to reconsider its confidence assessment
✓ Correct — Explanation
The problem is over-reliance on self-reported confidence. The fix is deterministic rule-based escalation: define the exact conditions that require human judgment in your routing logic, derive them from structured extraction fields, not Claude's uncertainty expressions. This reduces false escalations (Claude saying "not confident" when it actually can handle it) while maintaining necessary escalations (rules fire when the actual conditions are met).


A customer support agent is routing all complex, multi-step issues to human agents based on Claude's self-reported uncertainty: "I'm not 100% confident about the billing policy here." 40% of these escalations turn out to be cases Claude could have resolved. What escalation signal should replace self-reported confidence?
A.Add a second AI model to verify and validate Claude's confidence assessment
B.Increase Claude's temperature parameter to reduce hedging uncertainty expressions in responses
C.Use programmatic signals: missing data fields, policy violations, error thresholds
D.Add "Be more confident in your assessments" to the agent's system prompt text
✓ Correct — Explanation
Programmatic escalation signals are reliable; self-reported confidence is not. Examples: (1) `get_customer` returned no policy tier → escalate because policy lookup is required. (2) Refund amount > $500 → escalate per policy. (3) Tool error count > 3 in this session → escalate. These conditions are objective, deterministic, and not subject to the model's calibration errors.


Information provenance in a multi-agent pipeline refers to:
A.Tracking which source (URL, document, record) each piece of information came from
B.The geographic location of the API servers processing the pipeline request for data residency compliance
C.The sequential order in which agents processed information in the pipeline
D.Legal provenance — ensuring AI content is not used in violation of any TOS
✗ Incorrect — Explanation
Information provenance is the ability to trace any claim in a synthesized output back to its original source. In a research pipeline, this means the synthesis output can identify which document, URL, or database record each fact came from. This enables fact-checking, transparency, and the ability to invalidate specific claims if a source is found to be unreliable.



A customer interacting with your support agent says: "I don't want to deal with a bot. Please connect me to a real person right now." The case involves a routine order status inquiry that the agent can easily resolve. What should the agent do?
A.Honor the request immediately and escalate to a human agent without attempting resolution
B.Resolve the order status inquiry first since it is simple, then offer to connect to a human
C.Explain that a human agent isn't available right now and attempt autonomous resolution
D.Ask the customer to confirm their preference for a human before escalating the case
✗ Incorrect — Explanation
Explicit customer requests for a human agent must be honored immediately, regardless of case complexity. The agent should not first attempt resolution — even on a simple case — when the customer has clearly stated their preference. Attempting to resolve first (A) ignores the customer's stated preference and damages trust. Saying a human isn't available (B) is deceptive if one is. Asking the customer to confirm (D) adds friction to an already expressed preference. The rule is: explicit human request = immediate escalation, no investigation first.


Your system uses prompt caching for a 50,000-token system prompt shared across all user requests. After a model upgrade, you notice cache hit rates dropped to near zero. What is the most likely cause?
A.The new model has a different context window size that invalidates caches
B.A small modification to the shared system prompt (even a single character) breaks the cache prefix match — the entire prefix must be byte-for-byte identical
C.Prompt caching is not available on the new model version
D.The 50,000-token system prompt exceeds the cache size limit
✓ Correct — Explanation
Prompt caching requires exact byte-for-byte match on the cached prefix. Any modification — a single extra space, a version string update, a date injection — breaks the cache match and forces full re-processing. After a model upgrade, review whether the system prompt was also updated. Cache hit rates near zero after an upgrade almost always indicate a prompt modification, not a model incompatibility.


Your legal document extraction system processes 500 contracts per hour. The pipeline has 99.1% availability but experiences occasional cascading failures when a malformed document causes the extraction model to generate a response that crashes the JSON parser, which in turn drops the next 30 documents in the queue. How should the pipeline be redesigned for resilience?
A.Add input validation to reject malformed documents before processing
B.Implement per-document error isolation: each document processed in an independent try-catch; validation failures set `status: "failed"` and log details without affecting the queue; circuit breaker pattern to prevent cascading failures
C.Process documents sequentially instead of in parallel to prevent cascade effects
D.Increase the retry count from 3 to 10 for all documents
✓ Correct — Explanation
Cascading failure prevention requires two patterns: (1) per-document error isolation — each document must fail independently without affecting others (try-catch per item, not per batch), and (2) the circuit breaker pattern — if error rates spike above a threshold, the pipeline temporarily stops accepting new documents and alerts operations rather than continuing to fail. Sequential processing would eliminate one cause of cascades but at a severe throughput cost.


You want to use Claude Code to refactor a monolithic authentication service into microservices. This is a week-long project. You plan a session architecture: Day 1 explores the codebase, Day 2-3 designs the new architecture, Day 4-5 implements. How should sessions be managed across this project?
A.Use a single long-running session for the entire week to preserve maximum context
B.Start a completely new session each day to avoid stale context
C.Use named sessions (`--resume <name>`) within each phase; at major phase transitions (exploration → design → implementation), start a new session with a structured summary of prior phase learnings and decisions
D.Fork the session at the start of each day to preserve rollback capability
✓ Correct — Explanation
Named session resumption within a phase preserves the context of ongoing work (exploration findings, design decisions). Phase transitions warrant new sessions with structured summaries: the implementation session starts with "Architecture decisions from design phase: [list]" rather than carrying all the exploratory dead-ends and discarded design options from days 1-3. This keeps each phase context focused and avoids stale reasoning from earlier phases.


Your data extraction pipeline achieves 97% overall accuracy across 10,000 documents. Based on this, your team decides to stop human review for all extractions with model confidence > 0.9. Three months later, you discover a systematic 40% error rate on one document type (scanned PDFs with handwritten annotations) that represents 8% of your volume. What went wrong?
A.The confidence threshold was set too high; lowering it to 0.8 would have caught more errors
B.Overall accuracy metrics can mask poor performance on specific document types or fields; stratified analysis by document type and field should have been done before automating high-confidence extractions
C.The 97% accuracy was calculated on a biased sample; use random sampling next time
D.Confidence scores above 0.9 are inherently unreliable; use a different metric
✓ Correct — Explanation
Aggregate accuracy metrics hide segment-level failures. 97% overall accuracy could coexist with 40% error on a specific document type if that type is a small fraction of volume — the high performance on the majority papers over the minority failures. The correct approach is stratified analysis: measure accuracy by document type, field, and other segments before reducing human review. Confidence threshold (A) doesn't address the root cause — the model was also highly confident on the scanned PDFs it got wrong. Sampling bias (C) is not the issue. Confidence score reliability (D) misidentifies the problem.




A developer is using Claude Code for a multi-day migration project. After working for 3 days, they discover that a fundamental assumption in the migration approach is wrong, requiring a significantly different strategy. The current session has 200+ turns of context. What is the best way to proceed?
A.Continue in the current session and add a note about the wrong assumption
B.Use `--resume` to jump back to the point before the wrong assumption was made
C.Start a new session with a structured summary of: (1) what was learned so far, (2) the corrected understanding, (3) the new migration strategy — discarding the stale approach from prior turns
D.Fork the current session to preserve the old approach while trying the new one
✓ Correct — Explanation
When a fundamental assumption is invalidated, the prior 200 turns of context based on the wrong assumption become actively misleading. Resuming or continuing in that context means Claude will reason from stale, incorrect premises. A new session with a structured summary that explicitly states the corrected understanding prevents stale reasoning. The summary should capture everything learned that is still valid, plus the corrected approach — a "clean start with memory" rather than a full restart.



Production monitoring reveals inconsistent synthesis quality. When aggregated results total ~75K tokens, the synthesis agent reliably cites information from the first 15K tokens (web search headlines and snippets) and the final 10K tokens (document analysis conclusions), but frequently omits critical findings that appear in the middle 50K tokens—even when those findings directly address the research question. How should you restructure the aggregated input?
A.Place a key findings summary at the beginning of the aggregated input and organize detailed results with explicit section headers for easier navigation.
B.Implement rotation that alternates which subagent's results appear first across different research tasks, ensuring both sources receive primacy positioning equally over time.
C.Stream subagent results to the synthesis agent incrementally, processing web search results first to completion before introducing document analysis findings.
D.Summarize all subagent outputs to under 20K tokens total before aggregation, ensuring content stays within the model's reliable processing range.
✗ Incorrect — Explanation
The pattern is classic lost-in-the-middle: reliable attention at the start and end, poor attention in the middle. Placing a key findings summary at the beginning leverages the primacy effect — the most critical information occupies the most reliably attended position. Explicit section headers throughout the 75K input help the model navigate and maintain attention on middle sections. These two changes directly mitigate the attention pattern without discarding any content. Rotation (B) distributes the problem across tasks but does not fix it — findings will still be missed, just from different positions. Streaming (C) creates sequential processing that eliminates the parallelism benefit. Aggressive summarization to 20K (D) risks losing important details that were in the original findings.


A multi-agent research system has a web search subagent and a document analysis subagent. Both return statistics on cloud adoption rates. The web search agent returns "48% of enterprises use cloud" from a 2021 report. The document analysis agent returns "81% of enterprises use cloud" from a 2024 report. The synthesis agent flags this as a conflict. What is the correct interpretation and what schema change prevents this misclassification?
A.This is a genuine conflict; annotate it as contested data and present both values without interpretation
B.The 2024 value supersedes the 2021 value; discard the older finding
C.These values represent the same metric at different points in time — a time series, not a conflict. Requiring subagents to include a `publication_date` or `data_collection_date` field in structured outputs enables the synthesis agent to correctly interpret temporal differences rather than treating them as source conflicts
D.Average the two values to produce a best estimate of current adoption
✓ Correct — Explanation
The same metric measured at different times is a time series, not a contradiction. Cloud adoption rising from 48% in 2021 to 81% in 2024 is entirely plausible. Without timestamps in the structured output, the synthesis agent has no way to distinguish "two sources disagree" from "the same metric changed over time." The fix is schema-level: require each subagent's structured output to include a `publication_date` or `data_collection_date` field. With dates, the synthesis agent can correctly interpret the values as a temporal progression and present them as a trend. Treating it as a conflict (A) misrepresents the data. Discarding the older value (B) loses valid historical context.


A document analysis agent is processing a 200-page report and consistently misses details from pages 80–140. What is the root cause and fix?
A.Context window exceeded — switch to a model with a larger context window size
B.Attention dilution on middle content — split into per-section passes with integration
C.Document contains tables Claude cannot parse correctly — convert all tables to plain text format first
D.Pages 80-140 are less relevant — use retrieval to focus on important sections
✓ Correct — Explanation
The "missing middle" symptom is a classic attention dilution pattern. The fix is per-section passes: chunk the 200-page report into logical sections, process each section with full attention (the entire context is that one section), then run a separate synthesis pass over all section summaries. A larger context window does not fix attention dilution — it just moves the diluted zone further in.



What is attention dilution and when does it occur?
A.When Claude processes too many requests simultaneously and latency increases
B.When the prompt is too vague and Claude spreads attention across irrelevant topics
C.When middle content in a very long context receives less reliable model attention
D.When the context window is completely full and model responses get truncated
✓ Correct — Explanation
Attention dilution is the tendency for transformer models to give less reliable attention to content in the middle of very long contexts ("lost in the middle" problem). Content at the beginning and end of the context receives more reliable attention. The solution is to avoid stuffing too much into a single context — use per-section passes instead.


An agent has been exploring a large legacy codebase for 2 hours. You notice it starting to give inconsistent answers — referencing "the typical pattern in this codebase" rather than specific classes it found earlier, and suggesting approaches it already ruled out 30 minutes ago. What is happening and what is the fix?
A.The model is confused by the complexity of the codebase; restart with a cleaner system prompt setup
B.Context degradation — use a scratchpad file for key findings plus /compact to reduce
C.The agent is hitting rate limits causing inconsistent responses; add retry logic
D.This is normal for large codebases; use a model with a larger context window size
✓ Correct — Explanation
Context degradation is a documented pattern in extended sessions: as the context window fills with verbose tool results, the model loses reliable access to earlier specific findings and falls back on generic knowledge. The fix combines two techniques: (1) maintain a scratchpad file where the agent records key findings (class names, entry points, patterns) after each major discovery, referencing it for subsequent questions; and (2) use /compact to reduce context usage by condensing verbose tool output. Restarting (A) loses all progress. Rate limits (C) cause errors, not inconsistency. A larger context window (D) delays but doesn't solve context degradation.



A customer support agent calls `lookup_customer("Smith")` and receives three matching records: John Smith (account #1001), Jane Smith (#1002), and Robert Smith (#1003). What should the agent do?
A.Select the most recently active account as it is most likely the caller
B.Select the first match returned by the lookup tool as the default result
C.Process the request for all three accounts simultaneously to cover all bases
D.Ask the customer for an additional identifier like email or order number
✓ Correct — Explanation
When a lookup returns multiple matches, heuristic selection (most recent, first result) risks performing actions on the wrong customer account — applying a refund, updating contact info, or canceling an order for the wrong person. The correct pattern is to ask the customer for a disambiguating identifier that is unique to their account: email, order number, or a card's last 4 digits. This is a deterministic resolution that avoids both heuristic errors and the impracticality of processing all matching accounts. Processing all three accounts (D) would be incorrect and potentially harmful.


A customer contacts your support agent asking for a price match against a competitor. Your policy document covers price matching against your own historical prices but is completely silent on competitor price matching. The agent has the tools and authority to apply discounts. What should the agent do?
A.Apply the largest standard discount available as a goodwill gesture to retain
B.Deny the request since the policy doesn't explicitly authorize competitor matching
C.Ask the customer for proof of the competitor's price and then match autonomously
D.Escalate to a human — policy gaps are a canonical escalation trigger for agents
✓ Correct — Explanation
Policy gaps — situations where policy is ambiguous or silent on the customer's specific request — are a canonical escalation trigger, distinct from technical complexity. The agent should not apply discounts it's not authorized for (A), nor deny valid requests it's not authorized to deny (B), nor autonomously interpret a silent policy (D). When policy is silent, a human must make the call. This is a reliability principle: agents should escalate uncertainty about authorization, not make autonomous decisions that extend their authority.



When should you use prompt caching for cost optimization?
A.When the same prefix (system prompt, examples, document) is reused across calls
B.For any API call to reduce costs regardless of the prompt content or structure being repeated
C.When the user asks the same question multiple times in a single conversation
D.When the response is expected to be very long and consume many output tokens
✗ Incorrect — Explanation
Prompt caching stores the KV cache of a prompt prefix. Subsequent calls that share the same prefix pay the cache read price instead of re-processing those tokens. Maximum benefit comes from large, stable prefixes — a long system prompt, a big document, or a large set of few-shot examples that are reused across many requests.

The web search subagent returns results for only 3 of 5 requested source categories (competitor websites and industry reports succeeded, but news archives and social media feeds timed out). The document analysis subagent successfully processed all provided documents. The synthesis subagent must now produce a findings summary from this mixed-quality input. What's the most effective error propagation strategy?
A.Have the synthesis subagent request the coordinator retry the timed-out sources with extended timeouts before proceeding, ensuring complete data coverage before synthesis begins.
B.Structure the synthesis output with coverage annotations indicating which findings are well-supported versus which topic areas have gaps due to unavailable sources.
C.Have the synthesis subagent return an error to the coordinator indicating incomplete upstream data, triggering a full retry or task failure.
D.Proceed with synthesis using only the successful sources, generating output without indicating which data was unavailable.


A research pipeline processes 50-page documents. The team proposes switching to a model with a 200K token context window to process each document in one call, eliminating the per-section chunking. What is wrong with this reasoning?
A.200K token models are more expensive, making the pipeline cost-prohibitive
B.Context window size does not equal attention quality — a larger window moves the diluted zone but does not eliminate it; focused per-section passes still produce more reliable extraction
C.200K token models are slower and would violate the pipeline's latency SLA
D.The reasoning is correct — a 200K window would eliminate the need for chunking
✓ Correct — Explanation
This is a fundamental misconception the exam tests directly. A larger context window does NOT fix attention dilution — it just means the dilution zone is further from the ends. A 50-page document in a 200K window will still have attention issues in the middle sections. Per-section passes where each section is the primary focus of the entire context window produce more reliable results than dumping the whole document into a large window.



A multi-agent research pipeline has a web search agent and a document analysis agent that both find statistics on AI adoption rates. The web search agent finds "34% of enterprises have deployed AI in production" (McKinsey 2024). The document analysis agent finds "67% of enterprises have deployed AI in production" (Gartner 2024). Both sources are credible. How should the synthesis agent handle this?
A.Use the more recent source and discard the older one
B.Average the two values: report "~50% of enterprises have deployed AI in production"
C.Pick the more conservative estimate to avoid overstating adoption
D.Report both values with their source attributions and explicitly annotate the conflict, preserving both for the final report to distinguish contested from well-established findings
✓ Correct — Explanation
When credible sources conflict, the synthesis agent must not arbitrarily select one value, average them, or suppress the conflict. The correct approach is to annotate the conflict: include both values with their source attributions (McKinsey vs Gartner, 2024), note the discrepancy, and structure the final report to distinguish well-established findings from contested ones. This preserves information provenance and lets readers understand the uncertainty rather than presenting a false consensus. Arbitrarily selecting recency (A), averaging (B), or conservatism (C) all destroy information and misrepresent the evidence.



A coordinator agent receives partial results from a failed subagent: web search returned 3 sources before a timeout, document analysis completed successfully, synthesis was not attempted. How should the coordinator proceed?
A.Discard all results and re-run the entire pipeline
B.Proceed with synthesis using only the document analysis results, discarding the incomplete web search
C.Evaluate partial results: if 3 sources + document analysis are sufficient for the quality bar, proceed; otherwise targeted re-delegation to web search only (not the full pipeline)
D.Ask the user whether to proceed with partial results
✓ Correct — Explanation
Intelligent partial result handling is a coordinator skill. The coordinator should evaluate: are the partial results (3 web sources + full document analysis) sufficient for the required quality? If yes, proceed — no need to re-run anything. If no, targeted re-delegation to only the failed component (web search) is more efficient than full pipeline re-runs. Discarding all results wastes the successful document analysis.




